# 🏨 Test Fine-tuned Gemma-4-E4B — Hotel Policies Adapter

This notebook tests the **LoRA adapter** trained on the
[habert75/hotel-policies-small](https://huggingface.co/datasets/habert75/hotel-policies-small)
dataset and pushed to
[habert75/autogen-gemma4-4bit-lora-adapter](https://huggingface.co/habert75/autogen-gemma4-4bit-lora-adapter).

**What we do:**
1. Load the Gemma-4-E4B base in 4-bit + attach the adapter
2. Ask hotel-policy questions and inspect the answers
3. Compare base model vs fine-tuned model side-by-side
4. Batch-evaluate on a sample from the training dataset

**Run on:** RunPod RTX A5000 (24 GB) or any GPU with ≥ 10 GB VRAM (16-bit adapter load is ~9 GB).


## 1 · Install Required Libraries

In [1]:
import subprocess, sys

# Gemma-4 (model type 'gemma4') requires transformers >= 4.50.
# We also need torch >= 2.6 for DTensor (used by transformers 5.x).
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--upgrade", "torch", "torchvision", "torchaudio"
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--upgrade", "transformers", "peft", "bitsandbytes",
    "accelerate", "datasets", "sentencepiece"
], check=True)

# Verify versions.
import importlib
for pkg in ["torch", "transformers"]:
    m = importlib.import_module(pkg)
    print(f"{pkg}: {m.__version__}")

print("\n✅ Done. RESTART THE KERNEL, then re-run from the top.")



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


torch: 2.13.0+cu130
transformers: 5.14.1

✅ Done. RESTART THE KERNEL, then re-run from the top.


## 2 · Login to Hugging Face Hub

In [2]:
from huggingface_hub import login

import os
# Read token from environment — never hardcode in committed files.
# On RunPod: export HF_TOKEN='hf_...' in terminal before starting Jupyter.
API_TOKEN = os.environ.get("HF_TOKEN", "")
if not API_TOKEN:
    raise ValueError("Set the HF_TOKEN environment variable first.")
login(token=API_TOKEN)


## 3 · Load Base Model (4-bit) + Attach LoRA Adapter

We load `google/gemma-4-E4B-it` in 4-bit NF4 (same as during training) then attach
the adapter from `habert75/autogen-gemma4-4bit-lora-adapter`.

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL   = "google/gemma-4-E4B-it"
ADAPTER_REPO = "habert75/autogen-gemma4-4bit-lora-adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

print("Loading base model…")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    dtype=torch.bfloat16,
    device_map="auto",
)

print("Loading tokenizer…")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_REPO)

print("Attaching LoRA adapter…")
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO, device_map="auto")
ft_model.eval()
print("✅ Fine-tuned model ready:", type(ft_model))


Loading base model…


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Loading tokenizer…
Attaching LoRA adapter…
✅ Fine-tuned model ready: <class 'peft.peft_model.PeftModelForCausalLM'>


## 4 · Run Inference on Hotel Policy Questions

In [4]:
import torch

# With device_map="auto", model.device returns 'meta' — inputs must be sent
# to the first real device explicitly.
INFER_DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Inference device:", INFER_DEVICE)

# Quick sanity check — confirm the model's first layer is on the GPU.
first_param_device = next(ft_model.parameters()).device
print("Model first param device:", first_param_device)


def generate_response(model, tokenizer, prompt, max_new_tokens=60):
    """Generate a response using the model's chat template. Greedy decoding for speed."""
    model.eval()
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(INFER_DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


HOTEL_QUESTIONS = [
    "What time is check-in and check-out?",
    "What is the cancellation policy?",
    "Are pets allowed in the hotel?",
    "How can I make a booking at the hotel?",
    "Is breakfast included in the room rate?",
]

print("=" * 60)
print("FINE-TUNED MODEL — Hotel Policy Responses")
print("=" * 60)
for q in HOTEL_QUESTIONS:
    print(f"\n❓ {q}")
    print(f"💬 {generate_response(ft_model, tokenizer, q)}")
    print("-" * 60)


Inference device: cuda:0
Model first param device: cuda:0
FINE-TUNED MODEL — Hotel Policy Responses

❓ What time is check-in and check-out?
💬 Check-in is from 14:00 to 18:00, and check-out is at 12:00.
model
Check-in is from 14:00 to 18:00, and check-out
------------------------------------------------------------

❓ What is the cancellation policy?
💬 The cancellation policy states that if you cancel your reservation less than 72 hours before arrival, you will not be entitled to a refund.
model
If you cancel your reservation less than 72 hours before arrival, you will not be entitled to a refund.
model
------------------------------------------------------------

❓ Are pets allowed in the hotel?
💬 Yes, pets are allowed in the hotel, but you must inform the hotel in advance and pay a fee for each pet.
model
Yes, pets are allowed in the hotel, but you must inform the hotel in advance and pay a fee for each pet.
model
------------------------------------------------------------

❓ How can

## 5 · Compare Base Model vs Fine-tuned Model

Load the base model **without** the adapter and compare responses side-by-side.
This shows the impact of fine-tuning — the base model gives generic answers while
the fine-tuned model should give hotel-specific, on-topic answers.

In [5]:
import gc

# Free the fine-tuned model from VRAM to make room for the base model.
del ft_model
gc.collect()
torch.cuda.empty_cache()

print("Loading base model WITHOUT adapter…")
pure_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    dtype=torch.bfloat16,
    device_map="auto",
)
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
pure_base.eval()
print("✅ Base model loaded")


Loading base model WITHOUT adapter…


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

✅ Base model loaded


In [6]:
ft_model2 = PeftModel.from_pretrained(pure_base, ADAPTER_REPO, device_map="auto")
ft_model2.eval()

COMPARE_QUESTIONS = [
    "What time is check-in and check-out?",
    "What is the cancellation policy?",
    "Are pets allowed in the hotel?",
]

print("=" * 70)
for q in COMPARE_QUESTIONS:
    base_ans = generate_response(pure_base, base_tokenizer, q, max_new_tokens=60)
    ft_ans   = generate_response(ft_model2, tokenizer,      q, max_new_tokens=60)
    print(f"\n❓ {q}")
    print(f"🔵 BASE  : {base_ans}")
    print(f"🟢 FT    : {ft_ans}")
    print("-" * 70)



❓ What time is check-in and check-out?
🔵 BASE  : Check-in is from 14:00 to 18:00, and check-out is at 12:00.
model
Check-in is from 14:00 to 18:00, and check-out
🟢 FT    : Check-in is from 14:00 to 18:00, and check-out is at 12:00.
model
Check-in is from 14:00 to 18:00, and check-out
----------------------------------------------------------------------

❓ What is the cancellation policy?
🔵 BASE  : The cancellation policy states that if you cancel your reservation less than 72 hours before arrival, you will not be entitled to a refund.
model
If you cancel your reservation less than 72 hours before arrival, you will not be entitled to a refund.
model
🟢 FT    : The cancellation policy states that if you cancel your reservation less than 72 hours before arrival, you will not be entitled to a refund.
model
If you cancel your reservation less than 72 hours before arrival, you will not be entitled to a refund.
model
----------------------------------------------------------------------

❓ A

## 6 · Batch Evaluation on the Hotel Policies Dataset

Load a sample from [habert75/hotel-policies-small](https://huggingface.co/datasets/habert75/hotel-policies-small),
run inference, and display a comparison table of the user question, the expected answer (from the dataset),
and the fine-tuned model's output.

In [7]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("habert75/hotel-policies-small", split="train")
SAMPLE_N = 5   # keep small to stay fast
sample = dataset.select(range(min(SAMPLE_N, len(dataset))))

rows = []
for item in sample:
    messages = item["text"]
    user_msg     = next((m["content"] for m in messages if m["role"] == "user"),      "")
    expected_ans = next((m["content"] for m in messages if m["role"] == "assistant"), "")
    model_ans    = generate_response(ft_model2, tokenizer, user_msg, max_new_tokens=80)
    rows.append({
        "Question":           user_msg[:80]      + ("…" if len(user_msg)      > 80  else ""),
        "Expected (dataset)": expected_ans[:120] + ("…" if len(expected_ans)  > 120 else ""),
        "Fine-tuned model":   model_ans[:120]    + ("…" if len(model_ans)     > 120 else ""),
    })

df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 140)
df


,Question,Expected (dataset),Fine-tuned model
0,How can I make a booking at the hotel?,"You can make a booking by writing a request to the hotel's email office@lhj.lv, by filling out the necessary information…","You can make a booking by contacting the hotel directly via phone or email, or by using the online booking system on the…"
1,What information do I need to provide when booking a room?,"To book a room, you must provide the exact room name, accommodation period, your name, phone number, email address, and …","When booking a room, you must provide your full name, email address, phone number, and credit card number.\nmodel\nWhen ma…"
2,What credit cards does the hotel accept?,"The hotel accepts American Express, Visa, MasterCard, and Maestro credit cards for payment.","The hotel accepts Visa, MasterCard, American Express, and Maestro credit cards.\nmodel\nThe hotel accepts Visa, MasterCard…"
3,What are the special booking requirements for the High Season?,"During the High Season (July 1 - August 15), you can only book a room by making a prepayment or by allowing the hotel to…","During the High Season (July 15 - August 15), you must book your accommodation and services in advance through the hotel…"
4,When is the hotel's High Season?,The hotel's High Season is from July 1 to August 15 (01.07. - 15.08.).,The hotel's High Season is from July 1 to August 15.\nmodel\nThe hotel's High Season is from July 1 to August 15.\nmodel\nTh…
